In [6]:
import re
from datetime import datetime, timedelta

exam_schedule = {
    ("CS", "3"): "10 May 2026",
    ("CS", "5"): "15 May 2026",
    ("IT", "3"): "12 May 2026",
    ("IT", "5"): "18 May 2026",
    ("ME", "5"): "20 May 2026",
    ("CE", "3"): "14 May 2026",
    ("EE", "5"): "22 May 2026",
}

YEAR_TO_SEMESTERS = {
    "1": ["1", "2"],
    "first": ["1", "2"],
    "2": ["3", "4"],
    "second": ["3", "4"],
    "3": ["5", "6"],
    "third": ["5", "6"],
    "4": ["7", "8"],
    "fourth": ["7", "8"],
}

def extract_entities(query):
    """
    Extracts semester, course, and date entities from a query string.
    Returns a dict with keys: semester, course, date (all may be None).
    """
    entities = {"semester": None, "course": None, "date": None}

    sem_match = re.search(
        r'(?:(?:sem|semester)\s*(\d+))|(\d+)(?:st|nd|rd|th)?\s*(?:sem|semester)',
        query, re.IGNORECASE
    )
    if sem_match:
        entities["semester"] = sem_match.group(1) or sem_match.group(2)

    if not entities["semester"]:
        year_match = re.search(
            r'\b(first|second|third|fourth|1|2|3|4)\s*year\b',
            query, re.IGNORECASE
        )
        if year_match:
            key = year_match.group(1).lower()
            semesters = YEAR_TO_SEMESTERS.get(key, [])
           
            entities["semester"] = semesters[0] if semesters else None
            entities["_year_hint"] = key  # store for richer responses

 
    course_match = re.search(r'\b(CS|IT|ME|CE|EE|EC)\b', query, re.IGNORECASE)
    if course_match:
        entities["course"] = course_match.group(1).upper()


    date_match = re.search(r'\b(today|tomorrow)\b', query, re.IGNORECASE)
    if date_match:
        entities["date"] = date_match.group(1).lower()

    return entities


print("Entity extractor ready.")
print("Demo:", extract_entities("When is the third year CS exam?"))


Entity extractor ready.
Demo: {'semester': '5', 'course': 'CS', 'date': None, '_year_hint': 'third'}


In [7]:
class ConversationContext:

    def __init__(self):
        self.reset()

    def reset(self):
        self.entities   = {"semester": None, "course": None, "date": None}
        self.history    = []          # list of {"role": "user"|"bot", "text": str}
        self.pending_slot = None      # "semester" | "course" | None

    def merge(self, new_entities):
        for key in ("semester", "course", "date"):
            if new_entities.get(key) is not None:
                self.entities[key] = new_entities[key]

        if "_year_hint" in new_entities:
            self.entities["_year_hint"] = new_entities["_year_hint"]

    def is_followup(self, query):
 
        words = query.strip().split()
        connectors = {"for", "of", "about", "and", "what", "how", "in", "the"}
        starts_with_connector = words[0].lower() in connectors if words else False
        return (
            len(words) <= 6
            or starts_with_connector
            or self.pending_slot is not None
        )

    def add(self, role, text):
        self.history.append({"role": role, "text": text})

    def recent(self, n=4):
        return self.history[-n:]


ctx = ConversationContext()
print("ConversationContext ready.")
print("is_followup('For CS?')  →", ctx.is_followup("For CS?"))
print("is_followup('When is the SEM 5 CS exam?') →", ctx.is_followup("When is the SEM 5 CS exam?"))


ConversationContext ready.
is_followup('For CS?')  → True
is_followup('When is the SEM 5 CS exam?') → False


In [8]:
def generate_response(ctx):
    """
    Uses ctx.entities (already merged) to build a response.
    Sets ctx.pending_slot when information is still missing.
    """
    semester = ctx.entities.get("semester")
    course   = ctx.entities.get("course")

    if semester and course:
        ctx.pending_slot = None        
        key = (course, semester)
        if key in exam_schedule:
            date = exam_schedule[key]
            return (f"The {course} Semester {semester} exam is scheduled on "
                    f"{date}. Best of luck! 🎓")
        else:
            return (f"Sorry, I don't have the schedule for {course} Semester {semester} yet. "
                    f"Please check the official notice board.")

    elif semester and not course:
        ctx.pending_slot = "course"
        return (f"Got it — Semester {semester}. "
                f"Which department are you asking about? (e.g. CS, IT, ME, CE, EE)")

    elif course and not semester:
        ctx.pending_slot = "semester"
        return (f"Sure! Which semester of {course} do you want to know about? "
                f"(e.g. Semester 3, Semester 5)")

    else:
        ctx.pending_slot = None
        return ("Please mention the course and semester. "
                "For example: *'When is the SEM 5 CS exam?'*")


ctx.entities = {"semester": "5", "course": "CS", "date": None}
print(generate_response(ctx))

ctx.entities = {"semester": "3", "course": None, "date": None}
print(generate_response(ctx))


The CS Semester 5 exam is scheduled on 15 May 2026. Best of luck! 🎓
Got it — Semester 3. Which department are you asking about? (e.g. CS, IT, ME, CE, EE)


In [ ]:
def chatbot():
    ctx = ConversationContext()

    print("=" * 52)
    print("  🎓  Exam Query Chatbot  (multi-turn enabled)")
    print("=" * 52)
    print("  Ask about exam dates, e.g.:")
    print("    • 'When is the SEM 5 CS exam?'")
    print("    • 'When is the exam?'  →  'For third year CS'")
    print("  Type 'reset' to start over, 'exit' to quit.")
    print("=" * 52 + "\n")

    while True:
        try:
            user_input = input("You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nBot: Goodbye! Best of luck for your exams.")
            break

        if not user_input:
            continue

        if user_input.lower() == "exit":
            print("Bot: Goodbye! Best of luck for your exams.")
            break

        if user_input.lower() == "reset":
            ctx.reset()
            print("Bot: 🔄 Conversation reset. How can I help you?\n")
            continue

        ctx.add("user", user_input)
        new_entities = extract_entities(user_input)
        is_followup  = ctx.is_followup(user_input)
        if is_followup:
            ctx.merge(new_entities)
        else:
            ctx.entities = {"semester": None, "course": None, "date": None}
            ctx.merge(new_entities)
        response = generate_response(ctx)
        ctx.add("bot", response)

        print(f"Bot: {response}\n")

chatbot()
